In [1]:
import os
import pandas as pd
from prettytable import PrettyTable

from lib.uncertinay_rat import SimulateRat

/Users/felix/MSE/03_projects/MT/zz_code/04_experiments/03_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DS_FEVER = '../02_data/2026-05-11_base/fever'
DS_NQ = '../02_data/2026-05-11_base/nq'

DS_FEVER_SHUFFLE = '../02_data/2026-05-11_shuffle/fever'
DS_NQ_SHUFFLE = '../02_data/2026-05-11_shuffle/nq'

In [3]:
def get_retrieval_success(row):
    ret_set = set([f"{r['document_id']}:{r['index']}" for r in row['retrieved']])
    ref_set = set([f"{r['document_id']}:{r['index']}" for r in row['reference']])

    return set(ref_set) <= set(ret_set)

def set_abstain(row):
    return (row['generated_answer'] == 'I DO NOT KNOW') or (row['generated_answer'] == 'NOT ENOUGH INFO')

def set_task_success(row):
    return row['correct_answer']

def set_generator_success(row):
    if row['retriever_success'] == True:
        return row['task_success']
    else:
        return row['abstain']

In [4]:
results_all = {}
success_rates = {}
samples_all = {}

simulate = SimulateRat()

def load(base, g, experiment, shuffled=False):
    if experiment == '.DS_Store':
        return
    if g != 'qwen':
        return
    if experiment in ['oracle', 'empty']:
        return

    results = pd.read_json(f'{base}/{g}/{experiment}/results.json')
    dataset = base.split('/')[-1]

    results['dataset'] = dataset
    results['generator'] = g
    results['retriever_strategy'] = experiment
    results['shuffled'] = shuffled

    results['correct_query'] = True
    results['retriever_success'] = results.apply(get_retrieval_success, axis=1)
    results['abstain'] = results.apply(set_abstain, axis=1)
    results['task_success'] = results.apply(set_task_success, axis=1)
    results['generator_success'] = results.apply(set_generator_success, axis=1)

    key = f"{dataset}_{g}_{experiment}"
    if shuffled == True:
            key = f'{key}_shuffled'

    results_all[key] = results
    success_rates[key], samples_all[key] = simulate.compute_uncertainty(results)

for d in [DS_FEVER, DS_NQ]:
# for d in [DS_FEVER]:
    for g in os.listdir(f"{d}"):
        if g == '.DS_Store':
            continue

        for experiment in os.listdir(f"{d}/{g}/"):
            load(d, g, experiment, False)

for d in [DS_FEVER_SHUFFLE, DS_NQ_SHUFFLE]:
# for d in [DS_FEVER_SWITCH]:
    for g in os.listdir(f"{d}"):
        if g == '.DS_Store':
            continue

        for experiment in os.listdir(f"{d}/{g}/"):
            load(d, g, experiment, True)

out = pd.concat((df for df in results_all.values()), ignore_index=True)
# out = out.loc[out['generator'] == 'qwen']

In [5]:
out.groupby(['dataset', 'generator', 'retriever_strategy', 'shuffled'])[['retriever_success', 'abstain', 'task_success', 'generator_success']].mean().round(2)

retriever_success  abstain  \
dataset generator retriever_strategy shuffled                               
fever   qwen      dense              False                  0.58     0.21   
                                     True                   0.58     0.20   
                  hybrid             True                   0.63     0.12   
                  sparse             False                  0.53     0.18   
                                     True                   0.53     0.18   
nq      qwen      dense              False                  0.35     0.36   
                                     True                   0.35     0.36   
                  hybrid             True                   0.35     0.31   
                  sparse             False                  0.24     0.43   
                                     True                   0.24     0.43   

                                               task_success  generator_success  
dataset generator retriever_strategy shuffled                                   
fever   qwen      dense              False             0.75               0.73  
                                     True              0.75               0.73  
                  hybrid             True              0.83               0.69  
                  sparse             False             0.77               0.66  
                                     True              0.78               0.65  
nq      qwen      dense              False             0.24               0.50  
                                     True              0.24               0.50  
                  hybrid             True              0.27               0.45  
                  sparse             False             0.22               0.53  
                                     True              0.22               0.53

In [6]:
t = PrettyTable(field_names=['Dataset', 'Retriever Strategy', 'Shuffled', 'P(R=1)', 'P(A=1)', 'P(T=1)', 'P(G=1)'])

for experiment in sorted(success_rates.keys()):
    id = experiment.split('_')
    shuffled = len(id) > 3
    df = out.loc[(out['dataset'] == id[0]) & (out['generator'] == id[1]) & (out['retriever_strategy'] == id[2]) & (out['shuffled'] == shuffled)]

    t.add_row([
        id[0],
        id[2],
        'True' if shuffled else 'False',
        f"{success_rates[experiment]['r']['mean']:.2f}",
        f"{success_rates[experiment]['a']['mean']:.2f}",
        f"{success_rates[experiment]['t']['mean']:.2f}",
        f"{success_rates[experiment]['g']['mean']:.2f}"
    ]) 
 
t

Dataset,Retriever Strategy,Shuffled,P(R=1),P(A=1),P(T=1),P(G=1)
fever,dense,False,0.58,0.21,0.75,0.73
fever,dense,True,0.58,0.20,0.75,0.73
fever,hybrid,True,0.63,0.12,0.83,0.69
fever,sparse,False,0.53,0.18,0.77,0.66
fever,sparse,True,0.53,0.18,0.78,0.65
nq,dense,False,0.35,0.36,0.24,0.50
nq,dense,True,0.35,0.36,0.24,0.50
nq,hybrid,True,0.35,0.31,0.27,0.45
nq,sparse,False,0.24,0.43,0.22,0.53
nq,sparse,True,0.24,0.43,0.22,0.53


In [8]:
t = PrettyTable(field_names=['Dataset', 'Retriever Strategy', 'Shuffled', 'P(A=1|R=1)', 'P(A=1|R=0)', 'P(T=1|R=1,A=0)', 'P(T=1|R=0,A=0)'])

for experiment in sorted(success_rates.keys()):
    id = experiment.split('_')
    shuffled = len(id) > 3
    df = out.loc[(out['dataset'] == id[0]) & (out['generator'] == id[1]) & (out['retriever_strategy'] == id[2]) & (out['shuffled'] == shuffled)]

    t.add_row([
        id[0],
        id[2],
        'True' if shuffled else 'False',
        f"{success_rates[experiment]['a_r1']['mean']:.2f}",
        f"{success_rates[experiment]['a_r0']['mean']:.2f}",
        f"{success_rates[experiment]['t_r1_a0']['mean']:.2f}",
        f"{success_rates[experiment]['t_r0_a0']['mean']:.2f}"
    ]) 
 
t

Dataset,Retriever Strategy,Shuffled,P(A=1|R=1),P(A=1|R=0),"P(T=1|R=1,A=0)","P(T=1|R=0,A=0)"
fever,dense,False,0.03,0.46,0.96,0.90
fever,dense,True,0.03,0.45,0.96,0.91
fever,hybrid,True,0.03,0.28,0.96,0.92
fever,sparse,False,0.03,0.35,0.95,0.93
fever,sparse,True,0.02,0.34,0.95,0.93
nq,dense,False,0.06,0.52,0.49,0.26
nq,dense,True,0.06,0.52,0.49,0.26
nq,hybrid,True,0.06,0.44,0.51,0.28
nq,sparse,False,0.08,0.54,0.52,0.29
nq,sparse,True,0.07,0.55,0.52,0.30
